# Language-Model Fine-Tuning — DIMER E2E Tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/language-model-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/language-model-pipeline/blob/main/tutorials/language_model_finetuning_colab.ipynb)

**Profile:** `E2E`  
**Notebook specification:** DIMER Notebook Specification `1.0`

This notebook exercises the real `language-model-finetuner` execution components used by the deployable training image. It resolves the selected model through `lmpipeline.ModelRegistry`, prepares data through the production `finetuner.data` and `finetuner.masking` paths, loads and adapts the model through `finetuner.backends`, trains through `finetuner.training`, packages through `finetuner.artifacts`, and generates/reloads through `finetuner.inference`.

The notebook does **not** carry an independent trainer, masking implementation, model loader, or artifact publisher.

**What a successful run establishes:** the selected workflow executes against explicit immutable runtime-source revisions, produces optimization evidence, writes machine-readable predictions, packages a PEFT adapter, and proves the serialized adapter is present and behaviorally active after a fresh reconstruction.

It does **not** establish benchmark accuracy, safety, fairness, calibration, robustness, or production fitness.


## Prerequisites and data handling

- **Runtime:** Google Colab or Jupyter with a CUDA GPU. The default QLoRA path is intended for a T4-class or larger GPU.
- **Network:** the notebook downloads exact Python package versions, an immutable `language-model-pipeline` source revision, an immutable `language-model-finetuner` source revision, the pinned sample dataset, and the selected model revision.
- **BYOD:** choose `Bring Your Own Dataset` to upload `train.jsonl`, optional `validation.jsonl`/`val.jsonl`, and optional `test.jsonl`, or one ZIP containing those files.

**BYOD privacy boundary.** Uploaded dataset bytes remain in the notebook runtime and are not sent by this notebook to an external inference service. Network requests are still made for code, package, model, and sample acquisition. Do not place confidential, restricted, personal, or sensitive data in a hosted notebook unless you are authorized to do so.


## 1. Install exact dependencies and immutable production source

The notebook records two distinct source identities:

- the immutable `language-model-pipeline` revision that supplies shared contracts and notebook support;
- the immutable `language-model-finetuner` revision that supplies the production training and inference implementation.

These are intentionally separate from the notebook/PR revision. Release evidence records all three identities.


In [ ]:
%pip -q install transformers==5.16.1 tokenizers==0.23.2 huggingface-hub==1.30.0 peft==0.20.0 accelerate==1.14.0 bitsandbytes==0.49.0 safetensors==0.8.0 datasets==4.8.5 pandas==2.3.3 PyYAML==6.0.3 Jinja2==3.1.6
%pip -q install --no-deps git+https://github.com/kurtvalcorza/language-model-pipeline.git@8a9935c20f90d90f333ce2a191eedb001a1f0830
!rm -rf /content/language-model-finetuner
!git clone -q https://github.com/kurtvalcorza/language-model-finetuner.git /content/language-model-finetuner
!git -C /content/language-model-finetuner checkout -q 4db4338b28db2260060ca3642fd29e3ca5e19119


In [ ]:
import gc
import json
import shutil
import sys
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset

FINETUNER_ROOT = Path("/content/language-model-finetuner")
sys.path.insert(0, str(FINETUNER_ROOT / "src"))

from lmpipeline.errors import Code, DatasetError
from lmpipeline.tutorial_api import (
    assert_finetuner_checkout,
    assert_no_split_leakage,
    assert_tutorial_runtime,
    canonical_dataset_digest,
    normalize_records,
    resolve_tutorial_model,
    seed_everything,
    sha256_file,
    zip_directory,
)
from finetuner.artifacts import build_provenance, stage_artifact, verify_manifest
from finetuner.backends import (
    attach_adapter,
    load_base_model,
    load_tokenizer,
    trainable_parameter_summary,
)
from finetuner.config import TrainingConfig
from finetuner.data import (
    NormalizedSplits,
    dataset_digest,
    load_normalized_splits,
    tokenize_splits,
)
from finetuner.inference import (
    generate_reply,
    load_adapter_for_inference,
    verify_adapter_active,
)
from finetuner.masking import build_masked_example
from finetuner.training import train

PIPELINE_RUNTIME_REVISION = "8a9935c20f90d90f333ce2a191eedb001a1f0830"
FINETUNER_RUNTIME_REVISION = "4db4338b28db2260060ca3642fd29e3ca5e19119"
RUNTIME = assert_tutorial_runtime()
assert_finetuner_checkout(FINETUNER_ROOT, FINETUNER_RUNTIME_REVISION)
if not torch.cuda.is_available():
    raise RuntimeError("QLoRA requires a CUDA GPU. In Colab choose Runtime > Change runtime type > T4 GPU.")
print(json.dumps({
    "pipelineRuntimeRevision": PIPELINE_RUNTIME_REVISION,
    "finetunerRuntimeRevision": FINETUNER_RUNTIME_REVISION,
    "runtime": RUNTIME,
}, indent=2))


## 2. Resolve the canonical model and seed stochastic operations

`BASE_MODEL_KEY` is resolved by the repository registry. The notebook has no model registry of its own. The seed is applied before tokenizer/model/adapter construction. Remaining GPU/quantized-kernel variability is printed rather than hidden.


In [ ]:
BASE_MODEL_KEY = "qwen3-1.7b" # @param {type:"string"}
MAX_SEQUENCE_LENGTH = 512 # @param {type:"integer"}
EPOCHS = 1 # @param {type:"integer"}
LEARNING_RATE = 0.0002 # @param {type:"number"}
LORA_RANK = 8 # @param {type:"integer"}
LORA_ALPHA = 16 # @param {type:"integer"}
SEED = 42 # @param {type:"integer"}

DETERMINISM = seed_everything(SEED)
ENTRY = resolve_tutorial_model(
    BASE_MODEL_KEY,
    method="qlora",
    max_sequence_length=MAX_SEQUENCE_LENGTH,
)
TOKENIZER = load_tokenizer(ENTRY)
print(json.dumps(DETERMINISM, indent=2))
print({
    "modelKey": ENTRY.key,
    "modelId": ENTRY.model_id,
    "revision": ENTRY.revision,
    "license": ENTRY.license,
})


## 3. Load the default public sample or BYOD through the production data path

The default sample is pinned to an immutable dataset revision and is tutorial/sanity data, not benchmark evidence. For BYOD, `finetuner.data.load_normalized_splits` resolves and normalizes the upload through the same production path used by the deployable finetuner.

If validation is absent, `finetuner.data.tokenize_splits` derives a deterministic content-hash validation split. No row is silently truncated.


In [ ]:
DATA_SOURCE = "Sample: Filipino SFT" # @param ["Sample: Filipino SFT","Bring Your Own Dataset"]
SAMPLE_LIMIT = 120 # @param {type:"integer"}
WORK_DIR = Path("/content/language-model-tutorial")
shutil.rmtree(WORK_DIR, ignore_errors=True)
WORK_DIR.mkdir(parents=True)

if DATA_SOURCE == "Sample: Filipino SFT":
    dataset_id = "jpaulpoliquit/ph-sft-ai-authored-v1"
    dataset_revision = "8333699c6cc7296cc69cefc09def010851ded919"
    source_rows = [
        dict(row)
        for row in load_dataset(dataset_id, revision=dataset_revision, split="train")
    ]
    normalized = sorted(normalize_records(source_rows), key=lambda item: item.fingerprint())
    selected = []
    skipped_over_length = 0
    for item in normalized:
        if len(selected) >= SAMPLE_LIMIT:
            break
        try:
            build_masked_example(
                TOKENIZER,
                list(item.messages),
                line_number=item.line_number,
                max_sequence_length=MAX_SEQUENCE_LENGTH,
            )
        except DatasetError as exc:
            if exc.code == Code.DATASET_SEQUENCE_TOO_LONG:
                skipped_over_length += 1
                continue
            raise
        selected.append(item)
    NORMALIZED = NormalizedSplits(
        splits={"train": selected},
        source=f"{dataset_id}@{dataset_revision}",
        archive=None,
    )
    DATASET_DIGEST = canonical_dataset_digest(NORMALIZED.splits)
    DATASET_PROVENANCE = {
        "source": dataset_id,
        "revision": dataset_revision,
        "license": "apache-2.0",
        "usage": "tutorial-sanity-not-benchmark",
        "availableExamples": len(normalized),
        "selectedExamples": len(selected),
        "skippedOverLengthBeforeSelectionComplete": skipped_over_length,
    }
else:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No BYOD files were uploaded")
    upload_root = WORK_DIR / "upload"
    upload_root.mkdir()
    for name, payload in uploaded.items():
        (upload_root / Path(name).name).write_bytes(payload)
    NORMALIZED = load_normalized_splits(
        upload_root,
        workdir=WORK_DIR / "resolved",
    )
    DATASET_DIGEST = dataset_digest(
        upload_root,
        WORK_DIR / "digest-resolved",
    )
    DATASET_PROVENANCE = {
        "source": "BYOD",
        "transport": NORMALIZED.source,
        "archive": NORMALIZED.archive,
        "usage": "user-provided",
    }

assert_no_split_leakage(NORMALIZED.splits)
print({name: len(items) for name, items in NORMALIZED.splits.items()})
print("dataset digest:", DATASET_DIGEST)


## 4. Tokenize and mask through `finetuner.data` / `finetuner.masking`

Assistant-only supervision is computed by the production masking implementation using the model tokenizer's chat template. User/system/template tokens are masked with `-100`. Prefix instability or over-length examples fail rather than being silently accepted.


In [ ]:
SPLITS = tokenize_splits(
    NORMALIZED,
    tokenizer=TOKENIZER,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    validation_fraction=0.20,
    seed=SEED,
)
print("effective splits:", SPLITS.counts(), "validation derived:", SPLITS.validation_was_derived)
print("supervised tokens:", {
    "train": sum(item.supervised_token_count for item in SPLITS.train),
    "validation": sum(item.supervised_token_count for item in SPLITS.validation),
    "test": sum(item.supervised_token_count for item in SPLITS.test),
})


## 5. Load the production base-model path and record a deterministic baseline

`finetuner.backends.load_base_model` owns the QLoRA loading behavior. `finetuner.inference.generate_reply` owns generation. Greedy decoding is used for reproducible before/after probes.


In [ ]:
LOADED = load_base_model(
    ENTRY,
    method="qlora",
    device="cuda",
    tokenizer=TOKENIZER,
)
BASE_EMBEDDING_SIZE = LOADED.model.get_input_embeddings().num_embeddings
PROBE_PROMPTS = [
    "Ipaliwanag sa simpleng Filipino kung ano ang machine learning.",
    "Magbigay ng tatlong paraan para mabawasan ang basura sa opisina.",
]
BASELINE_OUTPUTS = [
    generate_reply(LOADED.model, TOKENIZER, prompt, decoding={"do_sample": False})
    for prompt in PROBE_PROMPTS
]
print({
    "dtype": LOADED.torch_dtype,
    "quantized": LOADED.quantized,
    "targetModules": LOADED.target_modules,
})
display(pd.DataFrame({"prompt": PROBE_PROMPTS, "base": BASELINE_OUTPUTS}))


## 6. Attach and train through the production finetuner

`finetuner.backends.attach_adapter` and `finetuner.training.train` are the same implementation used by the deployable finetuner. The configuration includes the production scheduler, warmup, early-stopping, restoration, and weight-decay fields even when the tutorial defaults leave optional controls disabled.

Loss and perplexity are optimization evidence, not task-quality evidence.


In [ ]:
MODEL = attach_adapter(
    LOADED,
    rank=LORA_RANK,
    alpha=LORA_ALPHA,
    dropout=0.05,
)
print("trainable parameters:", trainable_parameter_summary(MODEL))

TRAINING_CONFIG = TrainingConfig(
    method="qlora",
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    per_device_batch_size=1,
    gradient_accumulation_steps=2,
    seed=SEED,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    validation_split=0.20,
    weight_decay=0.01,
    lr_scheduler_type="constant",
    warmup_ratio=0.0,
    early_stopping_patience=0,
    early_stopping_min_delta=0.0,
    restore_best_adapter=False,
)
METRICS = train(
    MODEL,
    SPLITS,
    config=TRAINING_CONFIG,
    pad_token_id=TOKENIZER.pad_token_id,
    device="cuda",
).to_dict()
ADAPTED_OUTPUTS = [
    generate_reply(MODEL, TOKENIZER, prompt, decoding={"do_sample": False})
    for prompt in PROBE_PROMPTS
]
print(json.dumps(METRICS, indent=2))
display(pd.DataFrame({
    "prompt": PROBE_PROMPTS,
    "base": BASELINE_OUTPUTS,
    "adapted": ADAPTED_OUTPUTS,
}))


## 7. Run real new-prompt inference and export machine-readable outputs

`CUSTOM_PROMPT` is an editable new-input path. Results are written as JSONL plus a metrics JSON so downstream use does not depend on persisted notebook display state.


In [ ]:
CUSTOM_PROMPT = "" # @param {type:"string"}
NEW_PROMPTS = [
    "Sumulat ng maikling payo para sa isang estudyanteng nagsisimula sa AI.",
    "Ipaliwanag ang pagkakaiba ng training data at evaluation data sa dalawang pangungusap.",
]
if CUSTOM_PROMPT.strip():
    NEW_PROMPTS.append(CUSTOM_PROMPT.strip())

RESULT_ROWS = []
for index, prompt in enumerate(NEW_PROMPTS, 1):
    with MODEL.disable_adapter():
        base_answer = generate_reply(
            MODEL,
            TOKENIZER,
            prompt,
            decoding={"do_sample": False},
        )
    adapted_answer = generate_reply(
        MODEL,
        TOKENIZER,
        prompt,
        decoding={"do_sample": False},
    )
    RESULT_ROWS.append({
        "inputId": f"prompt-{index}",
        "prompt": prompt,
        "base": base_answer,
        "adapted": adapted_answer,
        "decoding": {"doSample": False},
        "modelId": ENTRY.model_id,
        "modelRevision": ENTRY.revision,
    })

OUTPUT_JSONL = WORK_DIR / "tutorial_predictions.jsonl"
OUTPUT_JSONL.write_text(
    "\n".join(json.dumps(row, ensure_ascii=False) for row in RESULT_ROWS) + "\n",
    encoding="utf-8",
)
METRICS_JSON = WORK_DIR / "tutorial_metrics.json"
METRICS_JSON.write_text(json.dumps(METRICS, indent=2), encoding="utf-8")
display(pd.DataFrame(RESULT_ROWS)[["inputId", "prompt", "base", "adapted"]])
print("wrote", OUTPUT_JSONL, "and", METRICS_JSON)


## 8. Package through `finetuner.artifacts` and verify a fresh active adapter

The artifact is staged and hashed by the production artifact implementation. The notebook then discards the training model, reconstructs tokenizer + exact base revision + serialized adapter through `finetuner.inference`, and performs two independent checks:

1. LoRA B matrices in the reloaded artifact are non-zero.
2. Adapter-on logits differ from adapter-off logits on the **same reloaded model**.

That proves the serialized adapter is present and effectful without requiring text-identical generation.


In [ ]:
JOB_DICT = {
    "training": TRAINING_CONFIG.to_dict(),
    "tutorial": {
        "profile": "E2E",
        "notebookSpecVersion": "1.0",
    },
}
PROVENANCE = build_provenance(
    entry=ENTRY,
    job_dict=JOB_DICT,
    dataset_digest=DATASET_DIGEST,
    loaded_dtype=LOADED.torch_dtype,
    quantized=LOADED.quantized,
    target_modules=LOADED.target_modules,
    dimer_base_model=None,
)
PROVENANCE["dataset"] = DATASET_PROVENANCE
PROVENANCE["runtimeRevisions"] = {
    "pipeline": PIPELINE_RUNTIME_REVISION,
    "finetuner": FINETUNER_RUNTIME_REVISION,
}
PROVENANCE["determinism"] = DETERMINISM

ARTIFACT_DIR = WORK_DIR / "dimer-lm-adapter"
STAGE = stage_artifact(
    MODEL,
    TOKENIZER,
    output_dir=ARTIFACT_DIR,
    entry=ENTRY,
    provenance=PROVENANCE,
    metrics=METRICS,
    base_embedding_size=BASE_EMBEDDING_SIZE,
)
verify_manifest(STAGE.path)
ARTIFACT_ZIP = zip_directory(
    STAGE.path,
    WORK_DIR / "dimer-language-model-adapter.zip",
)
ARTIFACT_SHA256 = sha256_file(ARTIFACT_ZIP)
print("artifact SHA-256:", ARTIFACT_SHA256)

del MODEL, LOADED
gc.collect()
torch.cuda.empty_cache()

RELOADED_MODEL, RELOADED_TOKENIZER = load_adapter_for_inference(
    STAGE.path,
    entry=ENTRY,
    device="cuda",
    quantized=True,
)
ACTIVITY = verify_adapter_active(
    RELOADED_MODEL,
    RELOADED_TOKENIZER,
    prompt="Kumusta.",
)
RELOAD_OUTPUT = generate_reply(
    RELOADED_MODEL,
    RELOADED_TOKENIZER,
    "Kumusta! Sagutin sa isang maikling pangungusap.",
    max_new_tokens=32,
    decoding={"do_sample": False},
)
if not RELOAD_OUTPUT:
    raise RuntimeError("Fresh artifact reconstruction generated no output")
print("fresh reconstruction PASS:", ACTIVITY)
print("fresh output:", RELOAD_OUTPUT)


## Interpretation and limits

A complete run establishes execution of the selected model/data/training/artifact path through the real finetuner modules at the recorded immutable revisions. It also establishes that the serialized adapter is non-zero and changes model logits after reconstruction.

It does **not** establish task accuracy, factual correctness, safety, fairness, calibration, robustness, or production fitness. Release-grade status additionally requires a clean supported-runtime execution record tied to the notebook/PR head **and** the two runtime-source revisions printed in Section 1.
